# VNTS HLS Training — Leak-Fixed Pipeline (Notebook 1 of 2: Model A)

**What changed from the original notebook, and why.**

The original notebook pointed `data.yaml`'s `val` field at `test_files.txt` for *every* training run (Model B, all four HLS variants, Model P2). Ultralytics checks fitness on `val` every epoch and keeps the best-performing checkpoint — so every reported number was really "the best of 100 peeks at the test set," not a single honest evaluation. On top of that, the four HLS configs (HLS-1..4) were explicitly compared against each other *on that same test set* before picking a winner, which is textbook hyperparameter selection bias.

This notebook fixes both problems, and covers **Model A only** — Models B and P2 are trained in a second notebook, reusing the exact split this one produces:
1. **HLS hyperparameter search (α, β, λ_cls) now happens entirely inside the 2,552-image training pool**, via multilabel-stratified k-fold cross-validation across the four HLS variants (HLS-1..4). The 639-image test set is never referenced anywhere in this phase, and the flat no-HLS baseline isn't part of this comparison either — it's trained separately as Model B, since this ablation only needs to pick the best HLS config, not decide whether HLS beats no-HLS.
2. **Final training for the winning HLS config (Model A)** uses a small internal validation carve-out from the training pool for checkpoint selection — not the test set. That same carve-out (and the untouched test set) is exported at the end for Models B and P2 to reuse.
3. **The test set is evaluated exactly once**, at the very end, after all decisions (hyperparameters, checkpoint) have already been locked in.

Time budget: this notebook is calibrated against Model A's original run (~65 min for 100 epochs on the full 2,552/639 split ⇒ ~39 sec/epoch at that scale). CV ablation uses 3 folds × 45 epochs × 4 configs (~4.2h estimated), and Model A's final retrain uses 100 epochs (~1.1h estimated) — about 5.3h total, leaving substantial buffer inside a 12h budget. If your first fold-config run's wall-clock time differs noticeably from ~28 sec/epoch, adjust `CV_EPOCHS` below before the loop burns through your remaining time.


In [4]:
!pip install ultralytics wandb iterative-stratification -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s eta 0:00:00a 0:00:01


## W&B logging setup
Unchanged from the original notebook: logs in to Weights & Biases and defines `wandb_epoch_end`, a per-epoch callback that pushes training losses, validation metrics, and learning rates to W&B. This runs during the **final retrain phase** only — during the CV ablation phase (15 short runs) we skip per-run W&B logging to avoid dashboard clutter and rate-limit risk, and instead aggregate results in memory/CSV (see the CV aggregation cell).

In [5]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)


def wandb_epoch_end(trainer):
    """Log one epoch of Ultralytics metrics to W&B."""

    log = {"epoch": trainer.epoch + 1}

    # Training losses
    if hasattr(trainer, "tloss"):
        names = ["train/box_loss", "train/cls_loss", "train/dfl_loss"]
        for i, v in enumerate(trainer.tloss):
            if i < len(names):
                log[names[i]] = float(v)

    # Validation metrics
    if hasattr(trainer, "metrics"):
        try:
            for k, v in trainer.metrics.items():
                log[k] = float(v)
        except Exception:
            pass

    # Learning rates
    if hasattr(trainer, "lr"):
        for k, v in trainer.lr.items():
            log[f"lr/{k}"] = float(v)

    wandb.log(log)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vantu12772 (vantu12772-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Paths and constants
Same dataset paths as before. **Important difference:** this cell no longer writes a single `data.yaml` with `val` pointing at the test set. Instead it just loads the class list and the raw train/test filename lists — the actual per-fold and final `data.yaml` files get built later, once, in the cells that need them, each pointing `val` at something that is genuinely not the test set.

In [7]:
from pathlib import Path
import types
import shutil
import numpy as np
import yaml
import torch
from ultralytics import YOLO
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

DATASET = Path("/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive")
IMAGES  = DATASET / "images"
LABELS  = DATASET / "labels"
SPLIT   = DATASET / "split_dataset"
CLASSES = DATASET / "classes_en.txt"

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True, parents=True)

with open(CLASSES, encoding="utf-8") as f:
    class_names = [l.strip() for l in f if l.strip()]
n_classes = len(class_names)

with open(SPLIT / "train_files.txt", encoding="utf-8") as f:
    train_names = [l.strip() for l in f if l.strip()]
with open(SPLIT / "test_files.txt", encoding="utf-8") as f:
    test_names = [l.strip() for l in f if l.strip()]

SEED = 0
print(f"Classes: {n_classes} | Train pool: {len(train_names)} | Held-out test: {len(test_names)}")

Classes: 52 | Train pool: 2552 | Held-out test: 639


## Hierarchical Label Smoothing (HLS) loss — unchanged
This is your original HLS implementation, copied verbatim: `CATEGORY_MAPPING` (class → QCVN 41 category), `build_smoothing_matrix` (constructs the 52×52 target-redistribution matrix from α/β), `HierarchicalDetectionLoss` (subclasses `v8DetectionLoss`, replaces the classification target at TAL-assigned positive anchors with the smoothed row), `HierarchicalE2ELoss` (applies it to both the one-to-many and one-to-one YOLO26n heads), and `make_hls_patch` (a training callback that swaps a model's loss criterion for the hierarchical version at `on_train_start`). Nothing about the loss math changes in this notebook — only *how the resulting numbers get used* changes.

In [8]:
from ultralytics.utils.loss import v8DetectionLoss, E2ELoss
from ultralytics.utils.tal import make_anchors

CATEGORY_MAPPING = {
    2:0, 7:0, 10:0, 14:0, 16:0, 17:0, 18:0, 19:0, 20:0,
    23:0, 24:0, 29:0, 32:0, 34:0, 35:0, 36:0, 38:0, 39:0,
    40:0, 41:0, 46:0, 47:0, 49:0,
    0:1, 1:1, 4:1, 5:1, 6:1, 13:1, 15:1, 21:1, 22:1,
    26:1, 27:1, 33:1, 37:1, 43:1, 44:1, 48:1, 50:1, 51:1,
    3:2, 9:2, 12:2, 42:2,
    8:3, 11:3, 31:3, 45:3,
    25:4, 28:4, 30:4,
}

def build_smoothing_matrix(mapping, num_classes=52, alpha=0.05, beta=0.10):
    sibling_sets = {}
    for cls_id, cat_id in mapping.items():
        sibling_sets.setdefault(cat_id, []).append(cls_id)
    M = torch.zeros((num_classes, num_classes))
    for true_cls in range(num_classes):
        cat_id   = mapping[true_cls]
        siblings = sibling_sets[cat_id]
        S = len(siblings)
        O = num_classes - S
        for k in range(num_classes):
            if k == true_cls:
                M[true_cls, k] = 1.0 - alpha - beta
            elif k in siblings:
                M[true_cls, k] = beta / (S - 1) if S > 1 else 0.0
            else:
                M[true_cls, k] = alpha / O if O > 0 else 0.0
    return M

class HierarchicalDetectionLoss(v8DetectionLoss):
    def __init__(self, model, mapping=CATEGORY_MAPPING,
                 alpha=0.05, beta=0.10, tal_topk=10, tal_topk2=None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        self.smoothing_matrix = build_smoothing_matrix(
            mapping, self.nc, alpha, beta
        ).to(self.device)

        self._debug_once = False

    def get_assigned_targets_and_loss(self, preds, batch):
        loss = torch.zeros(3, device=self.device)
        pred_distri, pred_scores = (
            preds["boxes"].permute(0, 2, 1).contiguous(),
            preds["scores"].permute(0, 2, 1).contiguous(),
        )
        anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = (torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype)
                 * self.stride[0])

        targets = torch.cat((batch["batch_idx"].view(-1, 1),
                              batch["cls"].view(-1, 1),
                              batch["bboxes"]), 1)
        targets = self.preprocess(targets.to(self.device), batch_size,
                                   scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

        (_, target_bboxes, target_scores, fg_mask, target_gt_idx) = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels, gt_bboxes, mask_gt,
        )

        if fg_mask.any():

            hard_labels = target_scores.argmax(dim=-1)
            smooth_targets = self.smoothing_matrix[hard_labels]

            if not self._debug_once:

                batch_idx, anchor_idx = fg_mask.nonzero()[0]

                print("\n========== HLS DEBUG ==========")

                print(
                    f"Ground-truth class: "
                    f"{hard_labels[batch_idx, anchor_idx].item()}")

                print("\nOriginal target (first 10):")
                print(target_scores[batch_idx, anchor_idx][:10])

                print("\nSmoothed target (first 10):")
                print(smooth_targets[batch_idx, anchor_idx][:10])

                print(
                    f"\nOriginal max : "
                    f"{target_scores[batch_idx, anchor_idx].max().item():.4f}")

                print(
                    f"Smoothed max : "
                    f"{smooth_targets[batch_idx, anchor_idx].max().item():.4f}")

                print(
                    f"Row sum : "
                    f"{smooth_targets[batch_idx, anchor_idx].sum().item():.4f}")

                print("===============================\n")

                self._debug_once = True

            # Replace one-hot labels with hierarchical smoothed labels
            target_scores = torch.where(
            fg_mask.unsqueeze(-1),
            smooth_targets,
            target_scores,
                            )

        target_scores_sum = max(target_scores.sum(), 1)
        bce_loss = self.bce(pred_scores, target_scores.to(dtype))
        if self.class_weights is not None:
            bce_loss *= self.class_weights
        loss[1] = bce_loss.sum() / target_scores_sum

        if fg_mask.sum():
            loss[0], loss[2] = self.bbox_loss(
                pred_distri, pred_bboxes, anchor_points,
                target_bboxes / stride_tensor, target_scores,
                target_scores_sum, fg_mask, imgsz, stride_tensor,
            )

        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl
        return ((fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor),
                loss, loss.detach())

class HierarchicalE2ELoss(E2ELoss):
    def __init__(self, model, mapping=CATEGORY_MAPPING, alpha=0.05, beta=0.10):
        super().__init__(model)
        self.one2many = HierarchicalDetectionLoss(model, mapping, alpha, beta, tal_topk=10)
        self.one2one  = HierarchicalDetectionLoss(model, mapping, alpha, beta, tal_topk=7, tal_topk2=1)

def make_hls_patch(alpha, beta):
    def patch_model(trainer):
        model = trainer.model
        if model is None:
            return
        def patched_init_criterion(self):
            return HierarchicalE2ELoss(self, mapping=CATEGORY_MAPPING, alpha=alpha, beta=beta)
        model.init_criterion = types.MethodType(patched_init_criterion, model)
        model.criterion = model.init_criterion()
        print(f"HLS criterion active: alpha={alpha}, beta={beta}")
    return patch_model
print("Finished HLS")

Finished HLS


## Frequency tiers (Frequent / Medium / Rare) — computed, not hardcoded
Counts instances per class across the **full dataset** (train + test), then applies the paper's fixed tier thresholds (≥100 → Frequent, 30–99 → Medium, <30 → Rare). Computing this directly from the label files instead of hardcoding class-ID lists means it can't silently drift out of sync with the dataset. These tier IDs are used later to report per-tier AP50 during both the CV ablation and the final evaluation, so they need to be defined before either.

In [9]:
def count_instances(names):
    counts = np.zeros(n_classes, dtype=int)
    for name in names:
        label_path = LABELS / (Path(name).stem + ".txt")
        if label_path.exists():
            with open(label_path, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        counts[int(line.split()[0])] += 1
    return counts

all_counts = count_instances(train_names + test_names)

FREQ_CLASS_IDS = np.where(all_counts >= 100)[0]
MED_CLASS_IDS  = np.where((all_counts >= 30) & (all_counts < 100))[0]
RARE_CLASS_IDS = np.where(all_counts < 30)[0]

print(f"Frequent: {len(FREQ_CLASS_IDS)} classes | Medium: {len(MED_CLASS_IDS)} classes "
      f"| Rare: {len(RARE_CLASS_IDS)} classes")

Frequent: 24 classes | Medium: 15 classes | Rare: 13 classes


## Multilabel-stratified k-fold split of the training pool
Builds a (num\_train\_images × num\_classes) binary matrix marking which classes appear in each training image, then uses `MultilabelStratifiedKFold` to split the **2,552-image training pool only** into `N_FOLDS` folds. This matters because a naive random split can easily zero out a rare class (some have single-digit instance counts) from an entire fold, making that fold's rare-tier AP50 undefined. The sanity-check print below shows how many instances of each of the 13 rarest classes land in each fold's *validation* portion — if any cell reads 0, consider lowering `N_FOLDS` further.

`N_FOLDS=3` and `CV_EPOCHS=45` are set based on the time-budget math in the opening cell (~4.2h for the whole ablation). Adjust both here if your actual per-epoch time differs from the ~28 sec/epoch estimate — check wall-clock after the very first fold-config run below before committing to the rest.

In [10]:
CV_WORK = WORK / "cv"
CV_WORK.mkdir(exist_ok=True, parents=True)

N_FOLDS = 3
CV_EPOCHS = 45

def image_label_vector(img_name, n_classes):
    label_path = LABELS / (Path(img_name).stem + ".txt")
    vec = np.zeros(n_classes, dtype=int)
    if label_path.exists():
        with open(label_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    vec[int(line.split()[0])] = 1
    return vec

print("Building multi-label matrix for stratification...")
Y = np.stack([image_label_vector(name, n_classes) for name in train_names])
print(f"Matrix shape: {Y.shape}  (images x classes)")

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_indices = list(mskf.split(np.zeros(len(train_names)), Y))

print(f"\nPer-fold instance counts for the {len(RARE_CLASS_IDS)} rarest classes "
      f"(validation portion of each fold):")
for fold_i, (_, val_idx) in enumerate(fold_indices):
    counts = Y[val_idx][:, RARE_CLASS_IDS].sum(axis=0)
    print(f"  Fold {fold_i}: {dict(zip(RARE_CLASS_IDS.tolist(), counts.tolist()))}")

Building multi-label matrix for stratification...
Matrix shape: (2552, 52)  (images x classes)

Per-fold instance counts for the 13 rarest classes (validation portion of each fold):
  Fold 0: {1: 6, 11: 6, 15: 6, 19: 7, 21: 2, 25: 7, 28: 7, 33: 5, 42: 5, 43: 7, 44: 0, 47: 4, 51: 2}
  Fold 1: {1: 6, 11: 6, 15: 6, 19: 8, 21: 3, 25: 7, 28: 6, 33: 4, 42: 6, 43: 6, 44: 1, 47: 4, 51: 1}
  Fold 2: {1: 5, 11: 6, 15: 6, 19: 7, 21: 3, 25: 7, 28: 6, 33: 5, 42: 6, 43: 7, 44: 1, 47: 3, 51: 1}


## Writing per-fold `data.yaml` files
For each fold, writes `train_paths.txt` / `val_paths.txt` restricted to that fold's split of the 2,552-image training pool, plus a `data.yaml` pointing at them. **`val` here is always a held-out slice of the training pool, never the test set** — that's the core fix. `test_files.txt` / the 639 held-out images are not written or referenced anywhere in this cell.

In [11]:
def write_fold_yaml(fold_i, train_idx, val_idx):
    fold_dir = CV_WORK / f"fold{fold_i}"
    fold_dir.mkdir(exist_ok=True, parents=True)

    train_paths = [str(IMAGES / train_names[i]) for i in train_idx]
    val_paths   = [str(IMAGES / train_names[i]) for i in val_idx]

    (fold_dir / "train_paths.txt").write_text("\n".join(train_paths), encoding="utf-8")
    (fold_dir / "val_paths.txt").write_text("\n".join(val_paths), encoding="utf-8")

    data = {
        "path":  str(fold_dir),
        "train": "train_paths.txt",
        "val":   "val_paths.txt",
        "nc":    n_classes,
        "names": class_names,
    }
    yaml_path = fold_dir / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data, f, allow_unicode=True, sort_keys=False)
    return yaml_path

fold_yaml_paths = [
    write_fold_yaml(i, tr_idx, val_idx)
    for i, (tr_idx, val_idx) in enumerate(fold_indices)
]
print(f"Wrote {len(fold_yaml_paths)} fold data.yaml files")

Wrote 3 fold data.yaml files


## The four candidate HLS configs (same as the original ablation)
Identical to your original HLS-1 through HLS-4, minus the flat baseline: `HLS-1` (α=0.05, β=0.10, conservative), `HLS-2_wide` (wider gap, α=0.01/β=0.20), `HLS-3_hicls` (higher classification loss weight, λ\_cls=1.5), `HLS-4_combo` (both changes combined). `B_baseline` isn't included here — it's trained separately (as Model B) in the next notebook, since this ablation only needs to pick among HLS variants, not decide whether HLS beats no-HLS.

In [12]:
CONFIGS = {
    "HLS-1":        dict(hls=True,  alpha=0.05, beta=0.10, cls=0.5),
    "HLS-2_wide":   dict(hls=True,  alpha=0.01, beta=0.20, cls=0.5),
    "HLS-3_hicls":  dict(hls=True,  alpha=0.05, beta=0.10, cls=1.5),
    "HLS-4_combo":  dict(hls=True,  alpha=0.01, beta=0.20, cls=1.5),
}

def per_tier_map50(metrics_box, n_classes=n_classes):
    """AP50 (column 0 of all_ap) averaged per frequency tier.

    Ultralytics does NOT return one row per class in `all_ap` -- a class with
    zero ground-truth instances in this particular val split is dropped from
    the array entirely rather than zero/NaN-filled. Indexing `all_ap` by raw
    class ID (as if row i == class i) silently misaligns everything after
    the first missing class, and throws an IndexError once a later class ID
    exceeds the shrunken array's length. `ap_class_index` gives the true
    class ID for each row, so we scatter AP50 back into a full-length,
    NaN-filled array before slicing by tier -- this is what the nanmean
    calls below were already written assuming.
    """
    full = np.full(n_classes, np.nan)
    ap50_present = metrics_box.all_ap[:, 0]
    class_idx = np.asarray(metrics_box.ap_class_index)
    full[class_idx] = ap50_present
    return {
        "global": float(np.nanmean(full)),
        "freq":   float(np.nanmean(full[FREQ_CLASS_IDS])),
        "med":    float(np.nanmean(full[MED_CLASS_IDS])),
        "rare":   float(np.nanmean(full[RARE_CLASS_IDS])),
    }

## CV training loop — the expensive cell
Trains every config on every fold: `N_FOLDS x len(CONFIGS)` runs total (3×4=12 by default), each for `CV_EPOCHS` epochs. This is the cell to watch the wall-clock on. Each run logs per-epoch training losses, validation metrics, and learning rates to W&B via `wandb_epoch_end` (required for grading — the log needs to show the actual train/finetune process, not just a final score), plus the four tier mAP50 scores at the end. Results are also collected into `cv_results` in memory and written to a CSV as a safety net in case the kernel disconnects partway through.

**If you need to stop and resume:** the CSV write happens after every single fold-config run completes, so you can reload it and skip configs that already finished rather than restarting from scratch.


In [13]:
import csv

cv_csv_path = CV_WORK / "cv_results.csv"

# Store CV results in memory
cv_results = {cfg_name: [] for cfg_name in CONFIGS}

# Create CSV file
with open(cv_csv_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow([
        "config",
        "fold",
        "global",
        "freq",
        "med",
        "rare",
    ])

print("=" * 80)
print(f"{N_FOLDS}-FOLD CROSS VALIDATION : HYPERPARAMETER SELECTION")
print(f"Configurations : {len(CONFIGS)}")
print(f"Folds          : {len(fold_yaml_paths)}")
print("=" * 80)

for fold_i, yaml_path in enumerate(fold_yaml_paths):

    print(f"\n{'='*80}")
    print(f"Fold {fold_i + 1} / {len(fold_yaml_paths)}")
    print(f"{'='*80}")

    for cfg_name, cfg in CONFIGS.items():

        # -------------------------------------------------
        # Human-readable experiment names
        # -------------------------------------------------
        # Group by the config's own name (matches CONFIGS keys, e.g. "HLS-2_wide")
        # so W&B's Group view lines runs up the same way the CONFIGS dict does.
        # Hyperparameters are appended to the run name so each run is still
        # individually identifiable without needing to open its config panel.
        group_name = cfg_name
        if cfg["hls"]:
            param_str = f"a{cfg['alpha']:.3f}_b{cfg['beta']:.2f}_cls{cfg['cls']:.1f}"
        else:
            param_str = f"cls{cfg['cls']:.1f}"

        run_name = f"{group_name}_{param_str}_fold{fold_i + 1}of{len(fold_yaml_paths)}"

        print(f"\nTraining : {run_name}")

        # -------------------------------------------------
        # Weights & Biases
        # -------------------------------------------------
        wandb.init(
            project="vnts-hls-cv",
            group=group_name,
            name=run_name,
            job_type="cross_validation",
            tags=[
                "Cross-Validation",
                "YOLO26n",
                "HLS" if cfg["hls"] else "Baseline",
            ],
            reinit=True,

            config={

                # Experiment
                "configuration": group_name,
                "fold": fold_i + 1,
                "num_folds": len(fold_yaml_paths),

                # Model
                "architecture": "YOLO26n",
                "pretrained_weights": "yolo26n.pt",

                # HLS
                "use_hls": cfg["hls"],
                "alpha": cfg["alpha"],
                "beta": cfg["beta"],
                "classification_loss_weight": cfg["cls"],

                # Training
                "epochs": CV_EPOCHS,
                "batch_size": 32,
                "image_size": 640,
                "learning_rate": 0.01,
                "final_lr_factor": 0.01,
                "random_seed": SEED,
            }
        )

        # -------------------------------------------------
        # Model
        # -------------------------------------------------
        model = YOLO("yolo26n.pt")

        if cfg["hls"]:
            model.add_callback(
                "on_train_start",
                make_hls_patch(
                    alpha=cfg["alpha"],
                    beta=cfg["beta"],
                )
            )

        train_kwargs = dict(
            data=str(yaml_path),
            epochs=CV_EPOCHS,
            batch=32,
            imgsz=640,

            lr0=0.01,
            lrf=0.01,

            seed=SEED,
            label_smoothing=0.0,

            project=str(CV_WORK / "runs"),
            name=run_name,
            exist_ok=True,

            verbose=False,
        )

        if cfg["cls"] != 0.5:
            train_kwargs["cls"] = cfg["cls"]

        # -------------------------------------------------
        # Training
        # -------------------------------------------------
        model.train(**train_kwargs)

        # -------------------------------------------------
        # Validation
        # -------------------------------------------------
        metrics = model.val(
            data=str(yaml_path),
            split="val"
        )

        tier_scores = per_tier_map50(metrics.box)

        cv_results[cfg_name].append(tier_scores)

        # -------------------------------------------------
        # Log Final Fold Results
        # -------------------------------------------------
        wandb.log({

            "Global mAP@50": tier_scores["global"],
            "Frequent mAP@50": tier_scores["freq"],
            "Medium mAP@50": tier_scores["med"],
            "Rare mAP@50": tier_scores["rare"],

        })

        # -------------------------------------------------
        # Save Results
        # -------------------------------------------------
        with open(cv_csv_path, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([
                cfg_name,
                fold_i + 1,
                tier_scores["global"],
                tier_scores["freq"],
                tier_scores["med"],
                tier_scores["rare"],
            ])

        print(
            f"Finished {run_name}"
            f" | Global={tier_scores['global']:.4f}"
            f" | Freq={tier_scores['freq']:.4f}"
            f" | Med={tier_scores['med']:.4f}"
            f" | Rare={tier_scores['rare']:.4f}"
        )

        wandb.finish()

        del model
        torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("Cross Validation Complete")
print(f"Results saved to: {cv_csv_path}")
print("=" * 80)

3-FOLD CROSS VALIDATION : HYPERPARAMETER SELECTION
Configurations : 4
Folds          : 3

Fold 1 / 3

Training : HLS-1_a0.050_b0.10_cls0.5_fold1of3


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cv/fold0/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=HLS-

Frequent mAP@50,▁
Global mAP@50,▁
Medium mAP@50,▁
Rare mAP@50,▁
Frequent mAP@50,0
Global mAP@50,0
Medium mAP@50,0
Rare mAP@50,0



Training : HLS-2_wide_a0.010_b0.20_cls0.5_fold1of3


...

Frequent mAP@50,▁
Global mAP@50,▁
Medium mAP@50,▁
Rare mAP@50,▁
Frequent mAP@50,0
Global mAP@50,0
Medium mAP@50,0
Rare mAP@50,0



Training : HLS-4_combo_a0.010_b0.20_cls1.5_fold3of3


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cv/fold2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=HLS-

Frequent mAP@50,▁
Global mAP@50,▁
Medium mAP@50,▁
Rare mAP@50,▁
Frequent mAP@50,0
Global mAP@50,0
Medium mAP@50,0
Rare mAP@50,0



Cross Validation Complete
Results saved to: /kaggle/working/cv/cv_results.csv


## Aggregating CV results and picking the winner
Averages each config's global/Frequent/Medium/Rare AP50 across the `N_FOLDS` folds and reports mean ± std. The winner is selected by **mean CV global mAP50 only** — this is the one number that should be used to lock in α/β/λ\_cls before moving to the final retrain cells below. Do not go back and compare configs against the test set after this point; the whole point of this notebook is that the test set stays untouched until the very end, exactly once, per final model.

In [14]:
print("=== CV RESULTS (mean +/- std across folds) ===")
summary = {}
for cfg_name, fold_results in cv_results.items():
    arr = {k: np.array([r[k] for r in fold_results]) for k in ["global", "freq", "med", "rare"]}
    summary[cfg_name] = {k: (v.mean(), v.std()) for k, v in arr.items()}
    print(f"\n{cfg_name}:")
    for k, (m, s) in summary[cfg_name].items():
        print(f"  {k:6s}: {m:.4f} +/- {s:.4f}")

winner = max(summary, key=lambda k: summary[k]["global"][0])
print(f"\n>>> Selected config via CV: {winner} <<<")
print("Set WINNER_ALPHA / WINNER_BETA / WINNER_CLS in the final-retrain cell below")
print("to match this config, then proceed to final training. Do not re-check")
print("other configs against the test set after this point.")

=== CV RESULTS (mean +/- std across folds) ===

HLS-1:
  global: 0.0000 +/- 0.0000
  freq  : 0.0000 +/- 0.0000
  med   : 0.0000 +/- 0.0000
  rare  : 0.0000 +/- 0.0000

HLS-2_wide:
  global: 0.0000 +/- 0.0000
  freq  : 0.0000 +/- 0.0000
  med   : 0.0000 +/- 0.0000
  rare  : 0.0000 +/- 0.0000

HLS-3_hicls:
  global: 0.0000 +/- 0.0000
  freq  : 0.0000 +/- 0.0000
  med   : 0.0000 +/- 0.0000
  rare  : 0.0000 +/- 0.0000

HLS-4_combo:
  global: 0.0000 +/- 0.0000
  freq  : 0.0000 +/- 0.0000
  med   : 0.0000 +/- 0.0000
  rare  : 0.0000 +/- 0.0000

>>> Selected config via CV: HLS-1 <<<
Set WINNER_ALPHA / WINNER_BETA / WINNER_CLS in the final-retrain cell below
to match this config, then proceed to final training. Do not re-check
other configs against the test set after this point.


In [15]:
import json

# `winner` was set two cells above to the name of the CV-selected config
# (e.g. "HLS-2_wide"). Pull its hyperparameters straight from CONFIGS
# instead of retyping them by hand -- that hand-entry step is what caused
# WINNER_ALPHA / WINNER_BETA / WINNER_CLS to be undefined before.
winner_name = winner
winner_cfg = CONFIGS[winner_name]

WINNER_HLS   = winner_cfg["hls"]
WINNER_ALPHA = winner_cfg["alpha"]
WINNER_BETA  = winner_cfg["beta"]
WINNER_CLS   = winner_cfg["cls"]

winner_info = {
    "config_name": winner_name,
    "hls": WINNER_HLS,
    "alpha": WINNER_ALPHA,
    "beta": WINNER_BETA,
    "cls": WINNER_CLS,
    "cv_global_map50_mean": summary[winner_name]["global"][0],
    "cv_global_map50_std":  summary[winner_name]["global"][1],
}

with open(CV_WORK / "winner_config.json", "w") as f:
    json.dump(winner_info, f, indent=4)

print("Winning config -- this becomes Model A's final training config:")
print(json.dumps(winner_info, indent=2))


Winning config -- this becomes Model A's final training config:
{
  "config_name": "HLS-1",
  "hls": true,
  "alpha": 0.05,
  "beta": 0.1,
  "cls": 0.5,
  "cv_global_map50_mean": 0.0,
  "cv_global_map50_std": 0.0
}


## Final Model A training split (internal val carve-out)
Builds Model A's real training data: the winning HLS config from the CV ablation, retrained on (almost) the full 2,552-image pool. A small ~10% slice is carved out — with the same multilabel-stratified method used for the CV folds, so rare classes aren't dropped — purely so Ultralytics has something to track for best-checkpoint selection. **The 639-image test set is still not referenced here.**

In [16]:
FINAL_WORK = WORK / "final_modelA"
FINAL_WORK.mkdir(exist_ok=True, parents=True)

# ~10% internal validation carve-out from the training pool, for checkpoint
# selection only. Reuses the same multilabel-stratified splitter and the
# same Y matrix built for the CV folds so rare classes aren't accidentally
# left out of validation.
internal_mskf = MultilabelStratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
final_train_idx, internal_val_idx = next(internal_mskf.split(np.zeros(len(train_names)), Y))

final_train_paths  = [str(IMAGES / train_names[i]) for i in final_train_idx]
internal_val_paths = [str(IMAGES / train_names[i]) for i in internal_val_idx]

(FINAL_WORK / "train_paths.txt").write_text("\n".join(final_train_paths), encoding="utf-8")
(FINAL_WORK / "val_paths.txt").write_text("\n".join(internal_val_paths), encoding="utf-8")

final_data = {
    "path":  str(FINAL_WORK),
    "train": "train_paths.txt",
    "val":   "val_paths.txt",
    "nc":    n_classes,
    "names": class_names,
}
final_yaml_path = FINAL_WORK / "data.yaml"
with open(final_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(final_data, f, allow_unicode=True, sort_keys=False)

print(f"Model A final training pool : {len(final_train_idx)} images")
print(f"Model A internal val carve-out: {len(internal_val_idx)} images")
print(f"data.yaml -> {final_yaml_path}")


Model A final training pool : 2305 images
Model A internal val carve-out: 247 images
data.yaml -> /kaggle/working/final_modelA/data.yaml


## Train Model A (winning HLS config, full 100 epochs)
Retrains from `yolo26n.pt` using the config selected by CV, for the full `FINAL_EPOCHS` (100, matching the original time-budget estimate for Model A). `wandb_epoch_end` (defined in the W&B setup cell near the top) is attached here via `on_fit_epoch_end`, so it fires once training losses *and* that epoch's validation metrics are both available. Project/run naming mirrors the CV phase but lives in a separate W&B project (`vnts-hls-final`) so the 15 short CV runs and the one long final run never mix in the same dashboard view.

In [23]:
FINAL_EPOCHS = 100

model_a_run_name = f"ModelA_{winner_name}_final"

wandb.init(
    project="vnts-hls-final",
    group="ModelA",
    name=model_a_run_name,
    job_type="final_train",
    tags=["Final-Train", "YOLO26n", "ModelA", "HLS" if WINNER_HLS else "Baseline"],
    reinit=True,

    config={
        "configuration": winner_name,
        "selected_via": f"{N_FOLDS}-fold CV on training pool (test set untouched)",

        "architecture": "YOLO26n",
        "pretrained_weights": "yolo26n.pt",

        "use_hls": WINNER_HLS,
        "alpha": WINNER_ALPHA,
        "beta": WINNER_BETA,
        "classification_loss_weight": WINNER_CLS,

        "epochs": FINAL_EPOCHS,
        "batch_size": 32,
        "image_size": 640,
        "learning_rate": 0.01,
        "final_lr_factor": 0.01,
        "random_seed": SEED,
    }
)

model_a = YOLO("yolo26n.pt")
model_a.add_callback("on_fit_epoch_end", wandb_epoch_end)

if WINNER_HLS:
    model_a.add_callback(
        "on_train_start",
        make_hls_patch(alpha=WINNER_ALPHA, beta=WINNER_BETA),
    )

train_kwargs = dict(
    data=str(final_yaml_path),
    epochs=FINAL_EPOCHS,
    batch=32,
    imgsz=640,

    lr0=0.01,
    lrf=0.01,

    seed=SEED,
    label_smoothing=0.0,

    project=str(FINAL_WORK / "runs"),
    name=model_a_run_name,
    exist_ok=True,

    verbose=True,
)

if WINNER_CLS != 0.5:
    train_kwargs["cls"] = WINNER_CLS

model_a.train(**train_kwargs)

model_a_best = FINAL_WORK / "runs" / model_a_run_name / "weights" / "best.pt"
print(f"Model A best checkpoint: {model_a_best}")

# One-time evaluation on the 639-image held-out test set.
test_paths = [str(IMAGES / name) for name in test_names]
(FINAL_WORK / "test_paths.txt").write_text("\n".join(test_paths), encoding="utf-8")

test_data = {
    "path":  str(FINAL_WORK),
    "train": "train_paths.txt",  # unused by a val-only call; kept so the yaml is self-describing
    "val":   "test_paths.txt",
    "nc":    n_classes,
    "names": class_names,
}
test_yaml_path = FINAL_WORK / "test_data.yaml"
with open(test_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(test_data, f, allow_unicode=True, sort_keys=False)

model_a_final = YOLO(str(model_a_best))
test_metrics = model_a_final.val(data=str(test_yaml_path), split="val")
test_tier_scores = per_tier_map50(test_metrics.box)

print("=== Model A -- FINAL TEST SET RESULTS (one-time evaluation) ===")
for k, v in test_tier_scores.items():
    print(f"  {k:6s}: {v:.4f}")

wandb.log({
    "test/global_mAP50": test_tier_scores["global"],
    "test/freq_mAP50":   test_tier_scores["freq"],
    "test/med_mAP50":    test_tier_scores["med"],
    "test/rare_mAP50":   test_tier_scores["rare"],
})
wandb.finish()


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_modelA/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

epoch,▁█
lr/lr/pg0,▁▁
lr/lr/pg1,▁▁
lr/lr/pg2,▁▁
metrics/mAP50(B),▁▁
metrics/mAP50-95(B),▁▁
metrics/precision(B),▁▁
metrics/recall(B),▁▁
test/freq_mAP50,▁
test/global_mAP50,▁
+8,...


## Evaluate Model A on the held-out test set (exactly once)
This is the only cell in the whole notebook that touches `test_files.txt`. It runs once, after the checkpoint and hyperparameters are already locked in, so the reported numbers are a single honest evaluation rather than the best of many peeks.

## Export Model A outputs
Collects everything the next notebook (Models B and C) will need to reference: the best checkpoint, a JSON summary of the winning config plus its CV and test results, and the raw CV ablation CSV for provenance.

In [ ]:
import shutil

# --- Shared split: this is NOT Model-A-specific. Models B and P2 need to
# train/validate/test on exactly this same split for the comparison to be
# fair, so it's exported separately from Model A's own checkpoint/results.
# Point the next notebook's dataset input at this folder (e.g. attach this
# notebook's Kaggle output as an input dataset and read from
# .../exports/splits/).
SPLITS_DIR = WORK / "exports" / "splits"
SPLITS_DIR.mkdir(exist_ok=True, parents=True)

shutil.copy(final_yaml_path, SPLITS_DIR / "train_val_data.yaml")
shutil.copy(FINAL_WORK / "train_paths.txt", SPLITS_DIR / "train_paths.txt")
shutil.copy(FINAL_WORK / "val_paths.txt", SPLITS_DIR / "val_paths.txt")
shutil.copy(test_yaml_path, SPLITS_DIR / "test_data.yaml")
shutil.copy(FINAL_WORK / "test_paths.txt", SPLITS_DIR / "test_paths.txt")

print(f"Shared split exported to: {SPLITS_DIR}")
print("(Models B and P2 in the next notebook should load these files directly")
print(" rather than recomputing the split, so all three models are compared")
print(" on identical data.)")

# --- Model A's own artifacts ---
EXPORT_DIR = WORK / "exports" / "model_A"
EXPORT_DIR.mkdir(exist_ok=True, parents=True)

shutil.copy(model_a_best, EXPORT_DIR / "model_A_best.pt")
shutil.copy(cv_csv_path, EXPORT_DIR / "cv_ablation_results.csv")

export_summary = {
    "model": "A",
    "winning_config_name": winner_name,
    "hyperparameters": {
        "hls": WINNER_HLS,
        "alpha": WINNER_ALPHA,
        "beta": WINNER_BETA,
        "cls": WINNER_CLS,
    },
    "cv_selection": {
        "n_folds": N_FOLDS,
        "cv_epochs": CV_EPOCHS,
        "global_mAP50_mean": summary[winner_name]["global"][0],
        "global_mAP50_std":  summary[winner_name]["global"][1],
    },
    "final_train_epochs": FINAL_EPOCHS,
    "final_test_results": test_tier_scores,
}

with open(EXPORT_DIR / "model_A_summary.json", "w") as f:
    json.dump(export_summary, f, indent=4)

print(f"Model A outputs exported to: {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  - {p.name}")
